# Multiple and Polynomial Regression

Multiple regression uses several features at once. Polynomial regression lets the line bend by adding powers of a feature.

## Imports and loading helper

In [ ]:
# Standard imports for the whole course
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Find the Datasets folder whether the notebook is run from the
# project root or from inside the notebooks folder.
from pathlib import Path
for candidate in [Path("Datasets"), Path("../Datasets")]:
    if candidate.exists():
        DATA = candidate
        break

def load(name):
    """Load a CSV from the Datasets folder by file name."""
    return pd.read_csv(DATA / name)

print("Datasets are in:", DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

## Multiple regression on the fifty companies data

State is categorical, so we one hot encode it, then drop one dummy column to avoid the dummy variable trap.

In [ ]:
df = load("Multiple_Linear_Regression_50_CompList.csv")
X = df.iloc[:, :-1].values
y = df.iloc[:, 4].values

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

le = LabelEncoder()
X[:, 3] = le.fit_transform(X[:, 3])
ct = ColumnTransformer([("State", OneHotEncoder(), [3])],
                       remainder="passthrough")
X = ct.fit_transform(X)
X = X[:, 1:]   # avoid the dummy variable trap

## Fitting and scoring the multiple regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
regressor = LinearRegression().fit(X_train, y_train)
y_pred = regressor.predict(X_test)

print("MAE :", metrics.mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
print("R2  :", regressor.score(X_test, y_test))

## Which feature matters most?

A correlation with the target ranks the numeric columns.

In [ ]:
print(df.corr(numeric_only=True)["Profit"])

## Polynomial regression on position salaries

We compare a straight line with polynomials of degree two and four.

In [ ]:
pos = load("Polynomial_Regression_Position_Salaries.csv")
Xp = pos.iloc[:, 1:2].values
yp = pos.iloc[:, 2].values

from sklearn.preprocessing import PolynomialFeatures

lin = LinearRegression().fit(Xp, yp)
xs = np.linspace(Xp.min(), Xp.max(), 100).reshape(-1, 1)

plt.scatter(Xp, yp, color="blue", zorder=3)
plt.plot(xs, lin.predict(xs), "r-", label="linear")
for deg, colour in [(2, "green"), (4, "orange")]:
    pf = PolynomialFeatures(degree=deg)
    pr = LinearRegression().fit(pf.fit_transform(Xp), yp)
    plt.plot(xs, pr.predict(pf.transform(xs)), color=colour,
             label=f"degree {deg}")
plt.xlabel("Position level"); plt.ylabel("Salary")
plt.legend(); plt.show()

Higher degree is not always better. Too low a degree underfits; too high a degree overfits.